# Task 10
#### #Build from the given skeleton

In [1]:
import datasets
import transformers
import numpy
from pprint import pprint

In [2]:
dset=datasets.load_dataset("imdb")

In [3]:
# make a tokenizer
base_tokenizer = transformers.AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokenizer = base_tokenizer.train_new_from_iterator(
    dset["train"]["text"],
    vocab_size=15000
)
tokenizer.tokenize(dset["train"][0]["text"])

['i',
 'rented',
 'i',
 'am',
 'curious',
 '-',
 'yellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 '.',
 'i',
 'also',
 'heard',
 'that',
 'at',
 'first',
 'it',
 'was',
 'se',
 '##ized',
 'by',
 'u',
 '.',
 's',
 '.',
 'custom',
 '##s',
 'if',
 'it',
 'ever',
 'tried',
 'to',
 'enter',
 'this',
 'country',
 ',',
 'therefore',
 'being',
 'a',
 'fan',
 'of',
 'films',
 'considered',
 '"',
 'controversial',
 '"',
 'i',
 'really',
 'had',
 'to',
 'see',
 'this',
 'for',
 'myself',
 '.',
 '<',
 'br',
 '/',
 '>',
 '<',
 'br',
 '/',
 '>',
 'the',
 'plot',
 'is',
 'centered',
 'around',
 'a',
 'young',
 'swedish',
 'drama',
 'student',
 'named',
 'lena',
 'who',
 'wants',
 'to',
 'learn',
 'everything',
 'she',
 'can',
 'about',
 'life',
 '.',
 'in',
 'particular',
 'she',
 'wants',
 'to',
 'focus',
 'her',
 'attention',
 '##s',
 'to',
 'making',
 'some

In [4]:
# Now we tokenize the IMDB dataset the usual way
def tokenize(ex):
    return {"tokenized":tokenizer.tokenize(ex["text"])}

dset=dset.map(tokenize,num_proc=4)

Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (704 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (642 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (752 > 512). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (593 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (515 > 512). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/50000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1021 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (616 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (696 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


In [5]:
from collections import Counter
from more_itertools import sliding_window #more-itertools is an awesome library!
import tqdm

def generate_ngrams(dset,n):
    for ex in tqdm.tqdm(dset):
        tokens=["<bos>"]*(n-1)+ex["tokenized"]+["<eos>"] # <--- insert enough <bos> tokens to ensure we always have history of length n-1, and insert one <eos> token
        for ngram in sliding_window(tokens,n):
            yield ngram


In [6]:
# Here we can concatenate all the individual datasets (train,test,unlabeled) in IMDB
# the "master" dataset is a dictionary of these, so dset.values() has the datasets of the individual sections (train,test,unlabeled)
combined_dataset=datasets.concatenate_datasets(list(dset.values()))

In [7]:
ngrams={} #This is the master dictionary
for ngram in generate_ngrams(combined_dataset,4): #let's start with 4-grams, you can try 3- and 5- grams too!
    prefix = ngram[:-1]
    word = ngram[-1]

    if prefix not in ngrams:
        ngrams[prefix]={}
    if word not in ngrams[prefix]:
        ngrams[prefix][word]=0
    ngrams[prefix][word] += 1
    


100%|██████████| 100000/100000 [00:30<00:00, 3239.16it/s]


In [8]:
def softmax(x):
    return numpy.exp(x)/sum(numpy.exp(x))

def sample_from(counts,temperature=1.0):
    """
    counts: list of counts that form the distribution
    temperature: the "how wild the generation should be" parameter, numbers close
                 to 0 are very conservative, numbers close or above 1 lead to quite
                wild generations
    """

    counts_array=numpy.array(counts)
    #Make these sum up to 1.
    counts_array_norm=counts_array/counts_array.sum()
    #Divide by temperature, that is what the algorithm does
    counts_array_norm/=temperature
    #Renormalize into a distribution using the softmax function, that is what the algorithm does
    final_distribution=softmax(counts_array_norm)
    #A good way to sample from a distribution is the following function from numpy
    x=numpy.random.multinomial(n=1,pvals=final_distribution)
    selected_word=numpy.argmax(x).flatten()
    return selected_word[0]

sample_from([1,1,1,17],temperature=0.5) #Try running this several times each, with temps 0.1, 0.5, 1.0 ... see how temp 0.1 sticks to picking the max value, but higher temps don't?

np.int64(3)

In [9]:
def generate(ngrams,n,max_len=40,temperature=1.0,prompt=None):
    """
    ngrams: the master dictionary
    n: the n in n-gram
    max_len: how many words max?
    temperature: the generation temperature
    prompt: the initial prompt, as a tuple, if not given n-1 <bos> symbols will be used
    """

    if prompt is None:
        prompt=["<bos>"]*(n-1) # <--- empty history means n-1 <bos> tokens

    generated=list(prompt) #this list will grow with words
    for _ in range(max_len):
        prefix=tuple(generated[-n+1:]) #pick the last n-1 from what we have generated so far
        d=ngrams[prefix] #the inner dictionary
        # Now we need to separate the words, and the counts for sampling
        # I do it with a less pythonic, more explicit code, could be done with list comprehensions of course
        possible_words=[] #list of words which could continue this ngram
        counts=[] #and their counts
        for word,count in d.items():
            possible_words.append(word)
            counts.append(count)
        new_word_index=sample_from(counts,temperature) #now sample which of the words gets selected, returns an index!
        new_word=possible_words[new_word_index]
        generated.append(new_word)

        if generated[-1]=="<eos>": #stop on end of sequence
            break
    return generated

# make sure to match the n below to the n which was used to create
# the master dictionary
for temp in (0.01,0.1,0.5,1.0,2.0,5.0):
    generated=generate(ngrams=ngrams,n=4,max_len=60,temperature=temp)
    print(f"Temp={temp}:")
    pprint(" ".join(generated))
    print("-----------")


Temp=0.01:
("<bos> <bos> <bos> i ' m not sure if this is the kind of movie that you can ' "
 "t stri ##ve to be realistic , it ' s a complement on the film , and i ' m "
 'not sure if it was a treasure . < br / > < br / > < br / > < br / >')
-----------
Temp=0.1:
('<bos> <bos> <bos> howling iv : the awakening . " if you knew a little more '
 'outrage ##d about this one . to get it all . and surely even if slightly py '
 '##rr ##hic victory . but who cares ? this is an often overlooked gem in 2005 '
 "and after almost one - and - corruption story , don ' t wed")
-----------
Temp=0.5:
('<bos> <bos> <bos> earnest effort which achieves some success to adapt the '
 "fairy tale atmosphere . it ' shocked ' me out of the hold and watched taker "
 'cho ##kes ##lam angle . rocky hit the rock bottom production values . i '
 'actually sympathize ##d with the gym , just seeing various events on new '
 'years of the silent comedians ( chaplin ,')
-----------
Temp=1.0:
('<bos> <bos> <bos> darn ##a ma